## STEP 1: Import Required Libraries

In [1]:
# Data handling
import pandas as pd
import numpy as np

# Model selection & validation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Evaluation metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## STEP 2: Load the Dataset

In [2]:
# Load the cleaned insurance fraud dataset
data = pd.read_csv("insurance_fraud_data_cleaned.csv")

# Separate features (X) and target variable (y)
X = data.iloc[:, :-1]   # All columns except last
y = data.iloc[:, -1]    # Last column = fraud label

## STEP 3: Train–Test Split

In [3]:
# Split data into training (80%) and testing (20%)
# Stratify ensures same fraud ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## STEP 4: Define Cross-Validation Strategy

In [4]:
# Stratified K-Fold ensures balanced class distribution in each fold
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## STEP 5: Define Advanced Models

In [5]:
# Dictionary of models to compare
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),      # Feature scaling
        ("model", LogisticRegression(max_iter=1000))
    ]),
    
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    
    "Random Forest": RandomForestClassifier(
        random_state=42
    )
}

## STEP 6: Cross-Validation (Model Stability Check)

In [6]:
print("\nCROSS-VALIDATION RESULTS (F1-Score)\n")

# Loop through models and perform 5-fold CV
for name, model in models.items():
    
    # cross_val_score returns score for each fold
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1"     # F1-score for imbalanced data
    )
    
    # Print average F1-score
    print(f"{name} → Mean F1-Score: {scores.mean():.4f}")


CROSS-VALIDATION RESULTS (F1-Score)

Logistic Regression → Mean F1-Score: 0.1963
Decision Tree → Mean F1-Score: 0.3618
Random Forest → Mean F1-Score: 0.1911


## STEP 7: Train Models on Full Training Data

In [7]:
trained_models = {}

# Train each model
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

## STEP 8: Evaluate Models on Test Dataset

In [8]:
print("\nTEST SET PERFORMANCE\n")

for name, model in trained_models.items():
    
    # Predict on test data
    y_pred = model.predict(X_test)
    
    # Print evaluation metrics
    print(name)
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1-score :", f1_score(y_test, y_pred))
    print("-" * 45)


TEST SET PERFORMANCE

Logistic Regression
Accuracy : 0.767304189435337
Precision: 0.7625
Recall   : 0.11030741410488246
F1-score : 0.19273301737756715
---------------------------------------------
Decision Tree
Accuracy : 0.6320582877959927
Precision: 0.3023255813953488
Recall   : 0.352622061482821
F1-score : 0.32554257095158595
---------------------------------------------
Random Forest
Accuracy : 0.773224043715847
Precision: 0.9365079365079365
Recall   : 0.10669077757685352
F1-score : 0.19155844155844157
---------------------------------------------


## STEP 9: Hyperparameter Tuning (Random Forest)

In [9]:
# Define parameter grid for tuning
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

# GridSearchCV tries all combinations using cross-validation
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring="f1",      # Optimize F1-score
    n_jobs=-1          # Use all CPU cores
)

# Fit grid search
grid.fit(X_train, y_train)

,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [None, 10, ...], 'min_samples_split': [2, 5], 'n_estimators': [100, 200]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


## STEP 10: Best Hyperparameters

In [10]:
print("\nBEST RANDOM FOREST PARAMETERS")
print(grid.best_params_)


BEST RANDOM FOREST PARAMETERS
{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}


## STEP 11: Evaluate Tuned Random Forest

In [11]:
# Get best tuned model
best_rf = grid.best_estimator_

# Predict on test data
y_pred_best = best_rf.predict(X_test)

# Final evaluation
print("\nTUNED RANDOM FOREST PERFORMANCE")
print("Accuracy :", accuracy_score(y_test, y_pred_best))
print("Precision:", precision_score(y_test, y_pred_best))
print("Recall   :", recall_score(y_test, y_pred_best))
print("F1-score :", f1_score(y_test, y_pred_best))


TUNED RANDOM FOREST PERFORMANCE
Accuracy : 0.77367941712204
Precision: 0.9375
Recall   : 0.10849909584086799
F1-score : 0.19448946515397084


🔹 1. What is Cross-Validation?
✅ Definition

Cross-validation is a technique used to evaluate how well a machine learning model generalizes to unseen data by training and testing it multiple times on different subsets of the dataset.

❓ Why Cross-Validation is Needed

A single train–test split:

Depends on random split

May give misleading results

Does not guarantee stability

Cross-validation solves this by:

Using multiple train-test splits

Producing reliable and stable performance metrics

🔁 How Cross-Validation Works (K-Fold)

Dataset is divided into K equal parts (folds)

Model is trained on K-1 folds

Tested on the remaining fold

Process repeats K times

Final performance = average of all runs

📌 Stratified K-Fold (Used in Your Project)

Maintains same class ratio in each fold

Essential for imbalanced datasets like fraud detection